In [3]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class RegularizedCNN(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.3):
        super(RegularizedCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.drop1 = nn.Dropout(dropout_rate)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.drop2 = nn.Dropout(dropout_rate + 0.1)

        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.bn_fc = nn.BatchNorm1d(256)
        self.drop_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x

Writing model.py


In [4]:
%%writefile train.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from model import RegularizedCNN

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Executing Training Pipeline on Device: {device}")

    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

    valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    valloader = DataLoader(valset, batch_size=64, shuffle=False, num_workers=2)

    model = RegularizedCNN(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    epochs = 15

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        epoch_train_loss = train_loss / total
        epoch_train_acc = correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in valloader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total

        scheduler.step(epoch_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc*100:.2f}%")

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print("--> Model Checkpoint Saved!")

if __name__ == '__main__':
    main()

Writing train.py


In [5]:
!python train.py

Executing Training Pipeline on Device: cuda
100% 170M/170M [26:46<00:00, 106kB/s]
Epoch [1/15] | Train Loss: 1.5969 Acc: 43.41% | Val Loss: 1.0468 Acc: 62.25%
--> Model Checkpoint Saved!
Epoch [2/15] | Train Loss: 1.1540 Acc: 58.64% | Val Loss: 0.8938 Acc: 68.16%
--> Model Checkpoint Saved!
Epoch [3/15] | Train Loss: 1.0072 Acc: 64.47% | Val Loss: 0.7724 Acc: 72.56%
--> Model Checkpoint Saved!
Epoch [4/15] | Train Loss: 0.9188 Acc: 67.45% | Val Loss: 0.6977 Acc: 75.68%
--> Model Checkpoint Saved!
Epoch [5/15] | Train Loss: 0.8638 Acc: 69.50% | Val Loss: 0.6993 Acc: 75.87%
Epoch [6/15] | Train Loss: 0.8134 Acc: 71.48% | Val Loss: 0.6555 Acc: 77.13%
--> Model Checkpoint Saved!
Epoch [7/15] | Train Loss: 0.7675 Acc: 73.21% | Val Loss: 0.5946 Acc: 79.22%
--> Model Checkpoint Saved!
Epoch [8/15] | Train Loss: 0.7352 Acc: 74.28% | Val Loss: 0.5634 Acc: 79.96%
--> Model Checkpoint Saved!
Epoch [9/15] | Train Loss: 0.7055 Acc: 75.54% | Val Loss: 0.5545 Acc: 80.76%
--> Model Checkpoint Saved!
E

In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


In [6]:
%%writefile evaluate.py
# ==============================================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from model import RegularizedCNN

def evaluate_performance():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    testloader = DataLoader(testset, batch_size=64, shuffle=False)

    model = RegularizedCNN(num_classes=10).to(device)
    model.load_state_dict(torch.load('best_model.pth', map_location=device))
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            y_true.extend(targets.numpy())
            y_pred.extend(predicted.cpu().numpy())

    classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

    print("================ Model Performance Report ================")
    print(classification_report(y_true, y_pred, target_names=classes))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title('CIFAR-10 Test Confusion Matrix — DGH2600170')
    plt.xlabel('Predicted Class')
    plt.ylabel('True Class')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300)
    plt.show()

if __name__ == '__main__':
    evaluate_performance()

Writing evaluate.py


In [7]:
!python evaluate.py

================ Model Performance Report ================
              precision    recall  f1-score   support

    airplane       0.83      0.85      0.84      1000
  automobile       0.93      0.92      0.93      1000
        bird       0.77      0.74      0.75      1000
         cat       0.74      0.65      0.69      1000
        deer       0.76      0.88      0.81      1000
         dog       0.81      0.72      0.76      1000
        frog       0.87      0.89      0.88      1000
       horse       0.91      0.87      0.89      1000
        ship       0.89      0.94      0.91      1000
       truck       0.87      0.92      0.90      1000

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000

Figure(1000x800)
